# 03. 메모리·STL·CMake — 60분

목표: Python의 리스트/딕셔너리와 C++ 컨테이너를 비교하고, 참조·포인터·수명을 설명하며 target 기반 라이브러리를 빌드합니다.
선행: C++ 1·2교시.

이 노트북은 **Python 커널** 에서 실제 C++ 도구를 호출합니다. GUI 없이 로컬·원격 모두 진행합니다.
코드는 아래 지정한 `exercise/` 파일을 JupyterLab 편집기로 수정합니다. 파일 저장 후 빌드 셀을 실행하세요.
노트북은 파일을 덮어쓰지 않습니다. 미완성 TODO의 검증 실패는 정상이며, 정답 확인 시점은 강사가 현장 상황에 맞춰 조절합니다.
정답은 별도 경로로 빌드하여 학습자 파일을 보존합니다. 빌드·결과는 호스트의 Data에 남습니다.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 이 노트북만으로 시작하기

열 파일은 `/workspace/03.C++/03_memory_stl_cmake.ipynb`입니다. [02_functions_classes.ipynb의 「값, 참조, const 참조 전달」](02_functions_classes.ipynb)에는 전달 방식 설명이 있으며 여기서 필요한 내용은 다음과 같습니다. 값 전달은 복사본, `T&`는 원본의 별칭, `const T&`는 원본을 바꾸지 않고 읽는 별칭입니다. `struct`는 관련 필드를 묶고 함수는 `return`으로 구조체를 돌려줄 수 있습니다. 이전 장바구니 소스·결과는 사용하지 않습니다.

첫 환경 셀이 `ROOT`, `LESSON`, `BUILD`, `run`, `build`를 정의합니다. 현재 노트북의 CMake 읽기→센서 TODO→빌드→CTest 순서대로 실행하세요. Jupyter 파일 편집기에서 `/workspace/03.C++/03_memory_stl_cmake/exercise/stats.cpp`를 수정하고 저장합니다. `print(read_text())` 셀은 파일을 보여줄 뿐 저장하지 않습니다. 결과는 `/workspace/Data/build/03.C++/03_memory_stl_cmake/exercise/`입니다.
<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

센서 통계 함수를 여러 파일로 나누면 어느 소스를 어떻게 빌드하고 어떤 검사를 실행할지 관리해야 합니다. vector는 입력 묶음을, 구조체는 통계 결과를 표현하고 CMake는 빌드 대상을 연결합니다. 통계 공식과 도구 실행은 서로 다른 확인 단계입니다.

### 전체 과정에서 내 코드가 하는 일

```text
stats.cpp TODO 수정·저장
  ↓
configure
  ↓
build
  ↓
CTest
  ↓
개수·평균·경보 결과 해석
```

통계 함수의 빈 입력·일반 입력·임계값 경계를 구현합니다. 설정 파일만 생성한 상태를 빌드 완료로 보지 않고, 테스트가 실제로 발견되어 실행됐는지도 확인합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 입력이 20·30·25이고 임계값이 30이면 평균과 경보는?

**설명:** 개수는 3, 평균은 25, 경보는 true입니다. 한 값 30이 임계값에 도달했기 때문입니다. 평균이 낮다고 경보를 끄면 과제의 판단 규칙이 달라집니다. 원본 vector 보존과 빈 입력 계약도 확인합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–4절의 수업 핵심 경로** → [1일차 슬라이드 81쪽](https://docs.google.com/presentation/d/1pQvfu3XzIfFL9FFvJkP8a6iNKJMjQFGOQIboEF6DRjg/edit#slide=id.d1_s076) · [배포 PDF 81쪽](../슬라이드/1일차.pdf#page=81)에서 88쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 제공된 빌드 틀에서 지정한 TODO를 수정하고 검증합니다.
- **단원 마지막 연결 질문:** Python 결과와 같음을 먼저 확인합니다. 기존 C++ 부품 재사용, 호출 비용, 수정·배포 부담 중 이 기능을 .so로 분리할 이유를 하나 선택합니다. C++이라는 이유만으로 빨라진다고 결론내리지 않습니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


## 처음 학습하는 학생을 위한 노트북 사용법

이 노트북의 설명은 위에서 아래로 읽되, 모든 문법을 외운 뒤 코드를 실행하려고 하지 않아도 됩니다. **코드 읽기 → 결과 예상 → 셀 실행 → 실제 결과 비교 → TODO 수정 → 검증** 순서로 반복하면 됩니다.

| 셀 또는 파일 | 하는 일 | 학생이 할 일 |
|---|---|---|
| Markdown 설명 | 새 용어와 코드의 목적 설명 | 낯선 용어를 표시하고 실행 결과를 먼저 예상 |
| 환경 Python 셀 | `g++`·CMake를 호출할 도우미와 경로 준비 | 내용을 바꾸지 말고 먼저 실행 |
| `examples/` | 완성된 작은 C++ 예제 | 출력 전에 변수 변화를 손으로 추적 |
| `exercise/` | 학생이 완성할 TODO | 지정된 부분만 수정하고 파일 저장 |
| build 셀 | 소스를 컴파일하고 링크 | 첫 오류부터 읽고 수정 후 다시 실행 |
| learner verification | 자신의 코드가 계약을 만족하는지 검사 | 정답 확인 시점은 강사가 조절 |
| 접힌 solution | 한 가지 참고 구현 | 통과 후 자신의 코드와 다른 이유를 비교 |

이 노트북의 커널은 Python입니다. Python 셀이 C++ 파일을 `g++` 또는 CMake에 전달하고 결과를 화면에 보여 줍니다. 마지막의 `CalledProcessError`만 읽지 말고, 그 위에 나온 컴파일러 또는 CTest의 **첫 번째 오류** 를 먼저 찾으세요. 공통 용어와 문제 해결 절차는 `03.C++/BEGINNER_GUIDE.md`에 더 자세히 정리되어 있습니다.

## 수업 필수와 복습·강사 시연 구분

| 구분 | 이 노트북에서 해당하는 절·셀 |
|---|---|
| **수업 필수** | 환경 셀, `vector`와 `const` 참조, 값/참조/포인터 차이, CMake configure/build/test, 센서 통계 계약표·TODO·CTest |
| **복습·강사 시연** | `array`·`tuple`·두 map의 세부 비교, 직접 `new[]/delete[]` 예제, CMake 전파 규칙 확장, 세 자습 노트북 안내 |

수업에서는 센서 통계 CTest 통과를 우선합니다. 여러 STL 타입과 직접 동적 할당은 강사가 실행 결과를 보여 주고 자습 노트북에서 반복할 수 있습니다.

## 진행표

| 시간 | 수업 핵심 |
|---|---|
| 0–5분 | 환경·목표 확인 |
| 5–15분 | vector·const 참조로 센서 값 읽기 |
| 15–25분 | 값·참조·비소유 포인터·수명 |
| 25–35분 | CMake target·소스/빌드 분리 |
| 35–55분 | 센서 요약 TODO·CTest |
| 55–60분 | 결과·자습 안내 |


In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '03.C++' / 'README.md').is_file())
LESSON = ROOT / '03.C++' / '03_memory_stl_cmake'
BUILD = ROOT / 'Data' / 'build' / '03.C++' / '03_memory_stl_cmake'
def run(args):
    result = subprocess.run([str(a) for a in args], text=True, capture_output=True, timeout=120)
    print(result.stdout, end='')
    if result.stderr: print(result.stderr, end='')
    result.check_returncode()
    return result

def build(variant):
    assert variant in ('exercise', 'solution')
    directory = BUILD / variant
    run(['cmake', '-S', LESSON, '-B', directory, '-DVARIANT='+variant, '-DCMAKE_BUILD_TYPE=Debug'])
    run(['cmake', '--build', directory, '--parallel', '2'])
    return directory
print('소스:', LESSON, '빌드:', BUILD)
run(['g++', '--version'])

import sys
print('Python:', sys.executable)


## 시작 전에 연결할 세 가지 관점

3교시는 컨테이너, 메모리, 빌드 시스템이라는 서로 달라 보이는 주제를 하나의 센서 통계 프로그램으로 연결합니다.

```text
센서 값 여러 개 저장 ──> vector가 원소와 길이를 소유
함수에 전달          ──> const 참조로 복사 없이 읽기
함수 구현 분리       ──> stats.cpp를 정적 라이브러리로 빌드
검증 프로그램 연결   ──> check_stats와 stats를 링크
여러 입력 검사       ──> CTest가 검증 프로그램 실행
```

여기서 **소유** 는 객체의 수명과 정리 책임을 관리한다는 뜻이고, **참조·포인터** 는 이미 존재하는 객체에 접근하는 방법입니다. 접근 수단이 있다고 해서 소유권이 생기는 것은 아닙니다. **target** 은 CMake가 만드는 실행 파일이나 라이브러리에 붙인 논리적 이름입니다.

수업 중에는 `vector`, 값/참조/포인터의 차이, configure/build/test 흐름을 우선 이해합니다. `array`, `tuple`, 두 종류의 map과 직접 동적 할당은 이 노트북에서 개념을 보고 자습 노트북으로 반복할 수 있습니다.

## 1. 자료구조를 고르는 기준

`vector`는 길이를 바꿀 수 있는 연속 배열입니다. `array<T,N>`은 크기가 타입에 고정됩니다. `tuple`은 서로 다른 타입의 필드를 묶습니다.
`map`은 키 순서로 순회하지만 `unordered_map`은 순서를 보장하지 않습니다. 둘 다 키로 값을 찾습니다.
`at()`은 범위를 검사하고 범위 밖 접근에 예외를 던집니다. `operator[]`는 같은 방식으로 검사하지 않습니다.

예측: 아래 map의 첫 출력 키는 무엇일까요? unordered_map 출력 순서로 정답을 판정해도 될까요?


In [ ]:
print((LESSON / 'examples/containers.cpp').read_text())
example_build = build('exercise')
run([example_build / 'containers'])


### 컨테이너 예제를 줄별로 추적하기

`examples/containers.cpp`는 서로 다른 목적의 컨테이너를 한 파일에서 비교합니다.

| 코드가 실행된 뒤 | 상태 또는 의미 |
|---|---|
| `samples{20, 30}` | `double` 두 개를 가진 vector, 크기 2 |
| `samples.push_back(25)` | 맨 뒤에 값이 추가되어 크기 3 |
| `shape{640, 480}` | 길이 2가 타입에 고정된 읽기 전용 array |
| `measurement{7, samples.front()}` | 정수 ID와 첫 온도 20.0을 하나의 tuple로 묶음 |
| `const auto [id, temperature]` | tuple의 두 값을 이름 있는 지역 변수로 풀어 읽음 |
| `counts{{"B",1},{"A",2}}` | map은 키 순서로 순회하므로 `A`, `B` 순서 |
| `lookup.find("A")` | 값을 삽입하지 않고 키 존재 여부를 확인 |

`size()`는 현재 원소 수이고 마지막 인덱스는 `size()-1`입니다. 빈 컨테이너에서는 `front()`, `back()`, `size()-1`을 사용하면 안 됩니다. `at()`은 범위를 확인해 예외를 내지만 `[]`는 범위 검사를 하지 않습니다.

`unordered_map`은 '정렬하지 않은 map'이라기보다 해시를 사용하는 별도 자료구조입니다. 순회 순서는 실행 환경이나 데이터 상태에 따라 달라질 수 있으므로 화면에 출력되는 순서로 정답을 판정하면 안 됩니다.

## 2. 주소·참조·포인터와 수명

값 전달은 복사본, 참조는 원본의 별칭, 포인터는 주소를 저장하는 값입니다.
`const T*`는 해당 포인터로 값을 바꾸지 못하게 할 뿐 메모리 수명을 연장하지 않습니다.
`new[]`로 할당한 배열은 `delete[]`로 해제하며, 해제 후 그 배열을 가리키는 모든 포인터를 사용하면 안 됩니다.
일반적인 배열 소유에는 범위 종료 시 정리되는 vector를 우선 사용합니다.

예측: `increase_value`, `increase_ref`, `*pointer += 2` 뒤 count는 얼마인가요?


In [ ]:
print((LESSON / 'examples/memory.cpp').read_text())
run([example_build / 'memory'])


### 같은 `count`를 바라보는 세 방법

`examples/memory.cpp`에서 변수와 메모리 관계는 다음과 같습니다.

```text
main의 count 객체 [1]
   │
   ├─ increase_value의 value [1]   별도 복사 → 함수 안에서 [2], count는 [1]
   ├─ increase_ref의 value ─────── count의 별명 → count가 [2]
   └─ pointer [count의 주소]       *pointer += 2 → count가 [4]
```

`&count`는 count의 주소를 얻고, `int* pointer`는 그 주소를 담습니다. `*pointer`는 주소가 가리키는 실제 정수에 접근합니다. `pointer`와 `*pointer`는 각각 주소와 그 주소의 값이므로 구분해야 합니다.

동적 배열 부분에서는 역할이 나뉩니다.

| 이름 | 역할 | 해제 책임 |
|---|---|---|
| `values` | `new double[3]`가 만든 배열을 소유하는 포인터 | `delete[] values`를 정확히 한 번 수행 |
| `view` | 같은 배열을 읽기 전용으로 바라보는 비소유 포인터 | 해제하면 안 됨 |
| `owned` | vector 객체가 내부 배열을 소유 | 범위를 벗어날 때 자동 정리 |

`delete[]` 뒤 주소 숫자가 남아 있더라도 배열 객체의 수명은 끝났습니다. 읽거나 쓰면 정의되지 않은 동작입니다. `nullptr`로 바꾸는 것은 이 포인터가 더 이상 대상을 가리키지 않는다고 표시할 뿐, 이미 복사해 둔 다른 포인터까지 고쳐 주지는 않습니다.

## 3. CMake는 빌드 대상을 연결합니다

<!-- procedure:lesson-05-dd54fb:start -->
![CMake로 빌드하고 검사하기](../교재그림/절차/lesson-05.svg)

빌드 오류와 검사 실패를 구분합니다. 산출물은 Data/build에, 수정 원본은 교재 폴더에 둡니다.
<!-- procedure:lesson-05-dd54fb:end -->


`add_library(stats STATIC ...)`는 재사용할 정적 라이브러리, `add_executable`은 실행 파일을 선언합니다.
`target_include_directories(stats PUBLIC ...)`의 PUBLIC은 소비하는 target에도 헤더 경로를 전달합니다.
`target_link_libraries(check_stats PRIVATE stats)`가 검증 프로그램과 라이브러리를 연결합니다.

configure(`cmake -S ... -B ...`)와 build(`cmake --build ...`)는 다른 단계입니다.
소스와 빌드를 분리하면 exercise·solution이 같은 파일을 덮어쓰지 않고 각각 빌드됩니다.


In [ ]:
print((LESSON / 'CMakeLists.txt').read_text())


### CMake 파일을 의존 그래프로 읽기

이 프로젝트의 빌드 관계는 다음과 같습니다.

```text
exercise/stats.cpp ──> [정적 라이브러리 stats] ──┐
                                                   ├─> [실행 파일 check_stats]
tests/check_stats.cpp ─────────────────────────────┘

examples/containers.cpp ──> [실행 파일 containers]
examples/memory.cpp ───────> [실행 파일 memory]
```

`cmake -S ... -B ...`의 configure 단계는 `CMakeLists.txt`를 읽어 위 관계에 맞는 빌드 규칙을 생성합니다. 아직 C++ 기계어를 모두 만드는 단계가 아닙니다. `cmake --build ...`가 변경된 소스를 컴파일하고 필요한 target을 링크합니다. 같은 build 명령을 다시 실행하면 보통 변경된 부분만 다시 만듭니다.

`target_include_directories(stats PUBLIC exercise)`는 `stats.cpp`뿐 아니라 stats를 링크하는 `check_stats`에도 헤더 검색 경로를 전달합니다. `target_link_libraries(check_stats PRIVATE stats)`는 check_stats가 stats 구현을 필요로 한다는 관계를 만듭니다. 헤더 선언을 읽는 것만으로 함수 구현이 실행 파일에 들어오는 것은 아니므로 링크 관계가 필요합니다.

CTest는 C++ 코드를 검사하는 특별한 컴파일러가 아닙니다. 빌드된 `check_stats` 프로그램을 실행하고 종료 코드를 수집하는 테스트 실행 도구입니다.

### [수업 필수] TODO를 풀기 전에 작성하는 계약 표

정답 코드를 보기 전에 다음 표를 종이에 채우고 구현 순서를 정하세요. 표는 구현 코드를 알려 주는 것이 아니라 함수가 지켜야 할 외부 약속을 분리합니다.

| 입력 종류 | `count`의 의미 | `mean`에서 확인할 점 | `alarm`에서 확인할 점 |
|---|---|---|---|
| 대표 입력 `{20,30,25}` | 원소 수 | 합과 원소 수의 관계 | 하나라도 기준 이상인지 |
| 빈 입력 | 0 | 0으로 나누지 않는 별도 규칙 | 경보가 켜지지 않아야 함 |
| 모두 음수 | 원소 수 | 정수 나눗셈이 아닌 실수 계산 | 기준보다 낮다면 false |
| 기준과 같은 한 값 | 1 | 그 값 자체 | `>`인지 `>=`인지 계약 확인 |

구현 후 CTest가 실패하면 다음 순서로 읽습니다.

1. configure가 실패했는지 build가 실패했는지 test가 실패했는지 구분합니다.
2. `summary_contract`가 Failed라면 위 입력 종류 중 어느 계약이 어긋났는지 봅니다.
3. 필요하면 터미널에서 `Data/build/03.C++/03_memory_stl_cmake/exercise/check_stats`를 직접 실행합니다.
4. 소스 파일을 저장하고 같은 exercise build 셀을 다시 실행합니다.

테스트는 반환 구조체 세 필드뿐 아니라 입력 vector가 바뀌지 않았는지도 확인합니다. `const std::vector<double>&` 계약과 일치해야 합니다.

## 4. 수업 핵심 TODO — 센서 통계

편집할 파일: **[03.C++/03_memory_stl_cmake/exercise/stats.cpp](03_memory_stl_cmake/exercise/stats.cpp)** .
`stats.hpp`에 선언된 Summary를 반환하세요. count는 개수, mean은 평균, alarm은 threshold 이상인 값이 하나라도 있는지입니다.
빈 배열은 `{0, 0.0, false}`를 반환하고 원본은 변경하지 마세요.

힌트: 빈 배열을 먼저 처리하고, 합계와 경보 상태를 반복문에서 누적합니다. 개수로 나눌 때 double 변환을 확인하세요.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `summarize(const std::vector<double>& values, double threshold)`는 읽기 전용 실수 목록과 경보 기준을 받아 `Summary{count, mean, alarm}`을 반환합니다. `{20,30,25}`, 기준 `30`이면 `{3,25.0,true}`입니다. 출력 문자열을 바꾸는 문제가 아닙니다.
**작성 흐름 힌트:** 헤더의 필드 순서 확인 → 빈 목록 반환 → 합계와 경보 누적 → 실수 평균 계산 → 구조체 반환. 평균은 합계/개수이고 경보는 평균과 무관하게 원소 하나라도 `>= threshold`이면 켜집니다.
**설명할 지점:** “평균이 25라서 정상이라고 말할 수 있을까요? 30인 값 하나를 놓치지 않는 별도 필드가 alarm입니다.”
<!-- todo-review:requirements:end -->

**선언과 성공 검사:** 현재 `exercise/stats.hpp`의 선언은 `struct Summary { int count; double mean; bool alarm; };`입니다. 반환 시 필드 순서는 개수→평균→경보이며 `values.size()`는 개수 타입이므로 `count`의 int와 평균 계산의 double 변환을 구분합니다. 유한한 온도 목록과 기준을 입력으로 가정합니다. `{}`는 `{0,0.0,false}`, `{-5,-1}`와 기준 `0`은 `{2,-3.0,false}`, `{30}`과 기준 `30`은 `{1,30.0,true}`입니다. 저장 후 바로 아래 exercise build와 CTest에서 `learner_verification=PASS`를 확인하세요.


In [ ]:
print((LESSON / 'exercise/stats.cpp').read_text())


In [ ]:
exercise_build = build('exercise')


In [ ]:
run(['ctest', '--test-dir', exercise_build, '--output-on-failure'])
print('learner_verification=PASS')


### 참고 정답

강사가 안내하는 시점에 열어 비교합니다. 실패하면 CTest 메시지를 확인하고 빈 입력·경계 온도·음수 입력을 다시 살펴보세요.


In [ ]:
print((LESSON / 'solution/stats.cpp').read_text())
solution_build = build('solution')


### 정답 해설 — 입력부터 결과까지

**처리 순서:** 빈 vector를 먼저 검사해 `{0,0.0,false}`로 끝냅니다. 값이 있으면 하나씩 합계를 더하고 `value >= threshold`인 값이 하나라도 있는지 누적합니다. 마지막에 개수와 평균·경보를 Summary로 돌려줍니다.

`[20,30]`, threshold=30이면 count=2, sum=50, mean=25.0, alarm=true입니다. 마지막 값이 낮더라도 앞에서 켜진 경보가 꺼지지 않도록 OR 조건으로 누적합니다. `const std::vector<double>&`는 원본을 빌려 읽는 계약입니다.

**빌드 흐름:** CMake는 stats 라이브러리를 만들고 check_stats 실행 파일에 연결합니다. configure는 규칙 준비, build는 컴파일·링크, CTest는 결과 검증입니다. 빌드 성공만으로 평균·경계가 맞았다고 결론내리지 않습니다.

**확인 질문:** 빈 입력을 먼저 처리하지 않으면 평균 계산에서 무엇이 잘못되나요?

**적용·재검증:** 자신의 답안을 저장한 뒤 지정된 exercise 파일/작성 셀에 반영하고 저장→exercise 빌드→학습자 검증 순서로 실행합니다. solution 빌드는 별도 참고 결과입니다. 오류는 첫 compiler/CTest 메시지부터 읽습니다. `.so`를 이미 로드했다면 재빌드 후 새 Python 프로세스 또는 커널 재시작과 환경 셀 재실행으로 다시 로드합니다. 결과는 기존 Data 경로에 저장합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](03_memory_stl_cmake/solution/stats.cpp)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```cpp
#include "stats.hpp"
Summary summarize(const std::vector<double>& values, double threshold) {
    if (values.empty()) return {0, 0.0, false};
    double sum{0.0};
    bool alarm{false};
    for (const double value : values) {
        sum += value;
        if (value >= threshold) alarm = true;
    }
    return {static_cast<int>(values.size()), sum / static_cast<double>(values.size()), alarm};
}
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
run(['ctest', '--test-dir', solution_build, '--output-on-failure'])
print('solution_verification=PASS')


## 완료·자습 확장

- mean이 낮아도 alarm이 True인 사례를 설명하세요.
- vector의 소유권과 `const vector<double>&` 접근의 차이를 설명하세요.
- 자습은 `90_selfstudy_stl.ipynb` → `91_selfstudy_memory.ipynb` → `92_selfstudy_cmake.ipynb` 순서입니다. 각각 약 60분이며 4시간 수업에 포함하지 않습니다.
- 자습 자료는 별도 lesson 폴더의 TODO를 사용하므로 이번 파일을 덮어쓰지 않습니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 센서 통계 구현·CMake 빌드·CTest 확인

[1일차 슬라이드 81쪽](https://docs.google.com/presentation/d/1pQvfu3XzIfFL9FFvJkP8a6iNKJMjQFGOQIboEF6DRjg/edit#slide=id.d1_s076) · [배포 PDF 81쪽](../슬라이드/1일차.pdf#page=81)로 돌아가 **81–88쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

